In [ ]:
!pip install transformers accelerate torch sentencepiece

Who is the model supposed to be?

In [ ]:
from transformers import AutoTokenizer, T5ForConditionalGeneration

# Load the model (runs on CPU)
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")
tok = AutoTokenizer.from_pretrained("google/flan-t5-small")

def ask(prompt):
    inputs = tok(prompt, return_tensors="pt", max_length=128, truncation=True)
    outputs = model.generate(**inputs, max_new_tokens=100)
    return tok.decode(outputs[0], skip_special_tokens=True)

print("=== EXAMPLE: WRITING A PROFESSIONAL EMAIL ===")

# Vague prompt with no context
print("\n--- NO CONTEXT ---")
no_context = ask("Write an email to a client about a delay.")
print(no_context)

# Detailed prompt with clear context
print("\n--- WITH CONTEXT ---")
with_context = ask("""Write a professional email to a client named Sarah Jones.
The project is delayed by 2 days because of a supplier issue.
Apologize sincerely, explain the reason briefly, and offer to schedule a quick call if she has concerns.
Keep the tone calm and solution-focused.""")
print(with_context)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


=== EXAMPLE: WRITING A PROFESSIONAL EMAIL ===

--- NO CONTEXT ---
I'm sorry to hear that. I'm sorry to hear that. I'm sorry to hear that. I'm sorry to hear that. I'm sorry to hear that. I'm sorry to hear that. I'm sorry to hear that. I'm sorry to hear that. I'm sorry to hear that. I'm sorry to hear that. I'm sorry to hear that. I'm sorry to hear that. I'm sorry

--- WITH CONTEXT ---
The project is delayed by 2 days because of a supplier issue. The project is delayed by 2 days because of a supplier issue. Apologize sincerely, explain the reason briefly, and offer to schedule a quick call if she has concerns.


What task should it perform?

In [ ]:
from transformers import AutoTokenizer, T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")
tok = AutoTokenizer.from_pretrained("google/flan-t5-small")

def ask(prompt):
    inp = tok(prompt, return_tensors="pt", max_length=256, truncation=True)
    out = model.generate(**inp, max_new_tokens=100)
    return tok.decode(out[0], skip_special_tokens=True)

question = "How do I reduce server costs?"

print("\n--- NO CONTEXT (Poor) ---")
print(ask(question))

print("\n--- SOME CONTEXT (Better) ---")
print(ask(f"Context: Cloud hosting\n{question}"))

print("\n--- MORE CONTEXT (Best) ---")
print(ask(f"Context: AWS cloud, 20 idle EC2 instances running\n{question}"))

print("\n💡 More context = Better performance!")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



--- NO CONTEXT (Poor) ---
Reduce server costs by reducing the amount of time you spend on the server.

--- SOME CONTEXT (Better) ---
Cloud hosting is a great way to reduce server costs.

--- MORE CONTEXT (Best) ---
Using a EC2 instance, you can reduce the server costs by reducing the server costs by reducing the server costs by reducing the server costs by reducing the server costs by reducing the server costs by reducing the server costs by reducing the server costs by reducing the server costs by reducing the server costs by reducing the server costs by reducing the server costs by reducing the server costs by reducing the server costs by reducing the server costs by

💡 More context = Better performance!


What background information should it use?

In [15]:
from transformers import pipeline

generator = pipeline('text-generation', model='gpt2')

def ask(prompt):
    result = generator(
        prompt,
        max_length=100,
        num_return_sequences=1,
        pad_token_id=50256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )
    # Print only the new generated part
    output = result[0]['generated_text'][len(prompt):]
    print(output.split('\n')[0])  # First line only

question = "Should we hire more developers?"

print("\n--- NO BACKGROUND ---")
ask(f"{question} Answer:")

print("\n--- BACKGROUND: Budget Crisis ---")
ask(f"Background: Lost $2M last quarter. Budget frozen.\n{question} Answer:")

print("\n--- BACKGROUND: Growing Fast ---")
ask(f"Background: Revenue up 300%. Can finish projects.\n{question} Answer:")

print("\n💡 Background information = Better decisions!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- NO BACKGROUND ---


Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Yes. The answer is, of course, that they need to make a living. The number of developers who are on the job, and the number of employees who are employed, are growing at a rapid rate, and the number of people with jobs is growing.

--- BACKGROUND: Budget Crisis ---


Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Absolutely.

--- BACKGROUND: Growing Fast ---
 No.

💡 Background information = Better decisions!


In [ ]:
from transformers import pipeline

generator = pipeline('text-generation', model='gpt2')

def ask(prompt):
    result = generator(prompt, max_length=80, do_sample=True, temperature=0.8, pad_token_id=50256)
    answer = result[0]['generated_text']
    print(answer + "\n")

print("="*60)
print("SAME QUESTION - DIFFERENT BACKGROUND - DIFFERENT ANSWER")
print("="*60)

question = "Recommendation:"

print("\n--- NO BACKGROUND ---")
ask(f"Question: Should I exercise today?\n{question}")

print("--- BACKGROUND: Injury ---")
ask(f"Background: I twisted my ankle yesterday. It's swollen and painful.\nQuestion: Should I exercise today?\n{question}")

print("--- BACKGROUND: Health Goal ---")
ask(f"Background: Doctor said I must exercise daily. I feel great today.\nQuestion: Should I exercise today?\n{question}")

print("💡 Background changes the recommendation!")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_length', 'temperature', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SAME QUESTION - DIFFERENT BACKGROUND - DIFFERENT ANSWER

--- NO BACKGROUND ---


Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Should I exercise today?
Recommendation:
The following are some suggestions (with additional quotes):
- Exercise is best during the day. Exercise is more important during the night.
- Do not go to bed early.
- Take a rest.
- Do not take up an activity.
- Exercise will keep you awake for a long time. Do not feel tired during exercise.
- Exercise is not stressful.
- Do not exercise when you are hungry. Exercise will be relaxing.
- Exercise is not necessary to work.
- Exercise is not necessary to maintain your weight. Exercise is not necessary to avoid pain.
- Do not exercise during your lunch or snack.
- Exercise will distract you.
- Exercise is not necessary to go on a date.
- Exercise is not necessary to sleep.
- Exercise will make you feel better.
- Exercise is not necessary to make you feel tired.
- Exercise is not necessary to feel sleepy.
- Exercise is not necessary to get more sleep.
- Exercise is not necessary to get more sleep. Exercise is important to maintain your sl

Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Background: I twisted my ankle yesterday. It's swollen and painful.
Question: Should I exercise today?
Recommendation: If you want to, talk to your doctor or even your physiotherapist.
Question: What do you think would be an appropriate massage?
Recommendation: Go to the doctor.
Question: If your doctor tells you what you're supposed to do, you should go do it now. You're not supposed to go to the gym.
Have a good day.
Maintain control and give yourself a good day.

--- BACKGROUND: Health Goal ---
Background: Doctor said I must exercise daily. I feel great today.
Question: Should I exercise today?
Recommendation: Exercise at least 60 minutes a day for 9 weeks.
Question: I'm getting tired of being up and about and being a bit over-excited. How can I get on with it?
Recommendation: Exercise at least eight hours a day for 8 weeks.
Question: I'm getting tired of feeling so rested with my feet. How can I get on with it?
In general, if the stress of getting to sleep is too much for you to ha

In [23]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
tok = AutoTokenizer.from_pretrained("google/flan-t5-base")

def ask(prompt):
    inp = tok(prompt, return_tensors="pt", max_length=512, truncation=True)
    out = model.generate(**inp, max_new_tokens=150, num_beams=4, do_sample=False)
    return tok.decode(out[0], skip_special_tokens=True)

def enforce_bullets(text):
    sentences = text.replace("\n", " ").split(". ")
    return "\n".join(f"- {s.strip()}" for s in sentences if s.strip())

def enforce_table(text):
    rows = []
    for part in text.replace("\n", " ").split(","):
        if ":" in part:
            metric, value = part.split(":", 1)
            rows.append(f"{metric.strip()} | {value.strip()}")
    header = "Metric | Value\n-------|------"
    return header + "\n" + "\n".join(rows)

content = "Our sales increased 50% in Q1. Customer satisfaction is at 95%. We launched 3 new products."

print("="*60)
print("SAME CONTENT - DIFFERENT FORMATS")
print("="*60)

print("\n--- FORMAT: 3 Bullet Points ---")
bp_raw = ask(f"Content: {content}\n\nFormat this as exactly 3 bullet points:")
print(enforce_bullets(bp_raw))

print("\n--- FORMAT: One Sentence Summary ---")
summary = ask(f"Content: {content}\n\nSummarize in one short sentence:")
print(summary)

print("\n--- FORMAT: Table ---")
table_raw = ask(f"Content: {content}\n\nFormat as a simple table with Metric and Value columns:")
print(enforce_table(table_raw))


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


SAME CONTENT - DIFFERENT FORMATS

--- FORMAT: 3 Bullet Points ---
- Our sales increased 50% in Q1
- Customer Satisfaction is at 95%
- We launched 3 new products

--- FORMAT: One Sentence Summary ---
Our sales increased 50% in Q1. Customer Satisfaction is at 95%. We launched 3 new products.

--- FORMAT: Table ---
Metric | Value
-------|------



In [18]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
tok = AutoTokenizer.from_pretrained("google/flan-t5-base")

def ask(prompt):
    inp = tok(prompt, return_tensors="pt", max_length=512, truncation=True)
    out = model.generate(**inp, max_new_tokens=150)
    print(tok.decode(out[0], skip_special_tokens=True))

topic = "Explain what machine learning is"

print("="*60)
print("SAME TOPIC - DIFFERENT AUDIENCE - DIFFERENT EXPLANATION")
print("="*60)

print("\n--- AUDIENCE: 8-year-old child ---")
ask(f"Audience: 8-year-old child\n\n{topic} in simple words:")

print("\n--- AUDIENCE: Business executive ---")
ask(f"Audience: Business executive with no tech background\n\n{topic} focusing on business value:")

print("\n--- AUDIENCE: Computer science student ---")
ask(f"Audience: Computer science graduate student\n\n{topic} with technical details:")

print("\n💡 Audience context changes complexity and tone!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


SAME TOPIC - DIFFERENT AUDIENCE - DIFFERENT EXPLANATION

--- AUDIENCE: 8-year-old child ---
Machine learning is a type of computer science that uses computer algorithms to learn information.

--- AUDIENCE: Business executive ---
Machine learning is a technology that enables businesses to make better decisions.

--- AUDIENCE: Computer science student ---
Machine learning is a technique for analyzing data and predicting the outcome of a computer program.

💡 Audience context changes complexity and tone!


In [24]:
"""
Shows: Accuracy, Relevance, Style Control, Task Alignment
"""

import subprocess, sys
print("📦 Installing...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers", "torch"])

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("🤖 Loading model (one time)...")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
tok = AutoTokenizer.from_pretrained("google/flan-t5-base")

def ask(prompt):
    inp = tok(prompt, return_tensors="pt", max_length=512, truncation=True)
    out = model.generate(**inp, max_new_tokens=100)
    print(tok.decode(out[0], skip_special_tokens=True))

print("\n" + "="*70)
print("1️⃣  ACCURACY - Model prioritizes right information")
print("="*70)
print("\n❌ Without context:")
ask("What should I do about memory issues?")

print("\n✅ With context:")
ask("Context: Python app using 8GB RAM, runs for 2 days then crashes.\nWhat should I do about memory issues?")

print("\n" + "="*70)
print("2️⃣  RELEVANCE - Avoids unrelated details")
print("="*70)
print("\n❌ Without context:")
ask("Tell me about Java")

print("\n✅ With context:")
ask("Context: I'm choosing a backend language for a web API.\nTell me about Java")

print("\n" + "="*70)
print("3️⃣  STYLE CONTROL - Guide tone and format")
print("="*70)
content = "Our app has 1M users. Revenue is $5M/year. Growing 20% monthly."

print("\n📱 Style: Social media post")
ask(f"Content: {content}\nWrite as an exciting tweet with emojis:")

print("\n📊 Style: Formal report")
ask(f"Content: {content}\nWrite as a formal business summary:")

print("\n" + "="*70)
print("4️⃣  TASK ALIGNMENT - Behaves as intended role")
print("="*70)
question = "Should I invest in cryptocurrency?"

print("\n❌ Generic response:")
ask(question)

print("\n✅ Role: Conservative financial advisor")
ask(f"Role: You are a conservative financial advisor focused on protecting client wealth.\n{question}")

print("\n✅ Role: Tech investor")
ask(f"Role: You are a tech investor who embraces calculated risks.\n{question}")

print("\n" + "="*70)
print("💡 Context transforms generic → task-specific responses!")
print("="*70)

📦 Installing...
🤖 Loading model (one time)...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



1️⃣  ACCURACY - Model prioritizes right information

❌ Without context:
Try to get a sedative.

✅ With context:
I'll try to use a different memory card.

2️⃣  RELEVANCE - Avoids unrelated details

❌ Without context:
Java is a small island in the Indian Ocean.

✅ With context:
Java is a programming language.

3️⃣  STYLE CONTROL - Guide tone and format

📱 Style: Social media post
@sadawnya i'm so excited!

📊 Style: Formal report
Our app has 1M users. Revenue is $5M/year. Growth is 20% monthly.

4️⃣  TASK ALIGNMENT - Behaves as intended role

❌ Generic response:
I.

✅ Role: Conservative financial advisor
(i)

✅ Role: Tech investor
No

💡 Context transforms generic → task-specific responses!


In [27]:
"""
ROLE CONTEXT
Tells the model who it should behave like
"""

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load FLAN-T5
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
tok = AutoTokenizer.from_pretrained("google/flan-t5-base")

def ask(prompt):
    inp = tok(prompt, return_tensors="pt", max_length=512, truncation=True)
    out = model.generate(**inp, max_new_tokens=120)
    print(tok.decode(out[0], skip_special_tokens=True))

question = "Explain quantum computing in simple terms."

print("\n--- WITHOUT ROLE ---")
ask(question)

print("\n--- ROLE: Physics Professor ---")
ask(f"As a physics professor, give a detailed lecture-style explanation with some technical depth.\n{question}")

print("\n--- ROLE: High School Teacher ---")
ask(f"As a high school teacher, explain this to teenagers using simple analogies and everyday examples.\n{question}")

print("\n--- ROLE: Tech Blogger ---")
ask(f"As a tech blogger, write a short blog post for a general audience with a catchy intro.\n{question}")

print("\n--- ROLE: Stand-up Comedian ---")
ask(f"As a stand-up comedian, explain quantum computing with humor and jokes.\n{question}")

print("\n💡 Role context changes style and tone!")


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



--- WITHOUT ROLE ---
quantum physics is the study of the physical properties of matter.

--- ROLE: Physics Professor ---
Quantum computing is a field of science that focuses on the physical properties of matter. Quantum computing is a field of science that focuses on the physical properties of matter.

--- ROLE: High School Teacher ---
Quantum computing is a process that uses a large number of variables to calculate the mass of a particle.

--- ROLE: Tech Blogger ---
Quantum computing is a term that describes the physical state of matter. It is a term that describes the physical state of matter. Quantum computing is a term that describes the physical state of matter. Quantum computing is a term that describes the physical state of matter. Quantum computing is a term that describes the physical state of matter. Quantum computing is a term that describes the physical state of matter. Quantum computing is a term that describes the physical state of matter. Quantum computing is a term tha

In [28]:
"""
INPUT DATA CONTEXT
Actual data the model must analyze
"""

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
tok = AutoTokenizer.from_pretrained("google/flan-t5-base")

def ask(prompt):
    inp = tok(prompt, return_tensors="pt", max_length=512, truncation=True)
    out = model.generate(**inp, max_new_tokens=100)
    print(tok.decode(out[0], skip_special_tokens=True))

print("\n--- WITHOUT DATA (Generic) ---")
ask("What trends do you see in our sales?")

print("\n--- WITH DATA (Specific Analysis) ---")
data = """
Jan: $2000
Feb: $2500
Mar: $3200
Apr: $4100
"""
ask(f"Analyze this sales data and identify trends:\n{data}")

print("\n💡 Data becomes part of reasoning!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



--- WITHOUT DATA (Generic) ---
Sales of our products are up by a third in the last year.

--- WITH DATA (Specific Analysis) ---
Jan - Feb - $2500 Mar - $3200 Apr - $4100

💡 Data becomes part of reasoning!
